In [2]:
import torch
import equiv_dens.utils.base as utils
import equiv_dens.utils.orbitals as orbitals
from equiv_dens.utils.spherical_harmonics import spherical_harmonics
import equiv_dens.utils.spherical_harmonics_deriv as sph_deriv
%load_ext autoreload
%autoreload 2

In [3]:
coords = torch.randn(1, 3, 3)
coords.requires_grad = True
print('coords', coords)
max_L = 5
d, u = utils.calculate_distances_and_directions(coords, center=torch.zeros(1, 1, 3))
s = spherical_harmonics(max_L, u)
print('s requires grad', [ss.requires_grad for ss in s])
for L in range(len(s)):
    zeros = torch.zeros_like(s[L])
    s[L] = torch.where(torch.isnan(s[L]), zeros, s[L])  # making sure there are no nans to avoid NaNs
print('dist', d)
print('unit', u)
scales = [torch.randn(1, 1, 1, 1) for L in range(max_L + 1)]
widths = [torch.randn(1, 1, 1, 1)**2 for L in range(max_L + 1)]
coeffs = [torch.randn(1, 1, 2 * L + 1, 1) for L in range(max_L + 1)]
print([ss.shape for ss in s])
print('scales', scales)
print('widths', widths)
print('coeffs', coeffs)

coords tensor([[[-1.4583,  0.5440, -1.0016],
         [-1.9079,  0.7346,  1.3076],
         [-1.0210,  0.8949,  0.2758]]], requires_grad=True)
s requires grad [False, True, True, True, True]
dist tensor([[[1.8509],
         [2.4268],
         [1.3854]]], grad_fn=<LinalgVectorNormBackward0>)
unit tensor([[[ 0.7879, -0.2939,  0.5411],
         [ 0.7861, -0.3027, -0.5388],
         [ 0.7370, -0.6460, -0.1991]]], grad_fn=<DivBackward0>)
[torch.Size([1, 3, 1]), torch.Size([1, 3, 3]), torch.Size([1, 3, 5]), torch.Size([1, 3, 7]), torch.Size([1, 3, 9])]
scales [tensor([[[[0.7699]]]]), tensor([[[[0.7861]]]]), tensor([[[[-1.4105]]]]), tensor([[[[0.9225]]]]), tensor([[[[0.3270]]]])]
widths [tensor([[[[0.0649]]]]), tensor([[[[0.3683]]]]), tensor([[[[0.1982]]]]), tensor([[[[6.4266]]]]), tensor([[[[0.2769]]]])]
coeffs [tensor([[[[-0.0412]]]]), tensor([[[[ 1.9022],
          [-0.7761],
          [ 0.7487]]]]), tensor([[[[ 0.1109],
          [ 0.0139],
          [ 0.7072],
          [ 0.2211],
      

In [36]:
gto = 0

for L in range(max_L + 1):
    scale = scales[L]
    width = widths[L]
    coeff = coeffs[L]
    sph = s[L].unsqueeze(-1) * coeff
    print('sph shape', sph.shape)
    # if L == 1:
        # print('sL shape', s[L].shape)
        # print('sph grad', torch.autograd.grad(sph[0, 0, 0, 0], coords))
    rbf = orbitals.gaussian_rbf(d.unsqueeze(-1), width, scale, L)
    gto += torch.sum(rbf * sph, dim=(-2, -1))
print(gto)

sph shape torch.Size([1, 3, 1, 1])
scale shape torch.Size([1, 1, 1, 1])
width torch.Size([1, 1, 1, 1])
r shape torch.Size([1, 3, 1, 1])
L 0
sph shape torch.Size([1, 3, 3, 1])
scale shape torch.Size([1, 1, 1, 1])
width torch.Size([1, 1, 1, 1])
r shape torch.Size([1, 3, 1, 1])
L 1
sph shape torch.Size([1, 3, 5, 1])
scale shape torch.Size([1, 1, 1, 1])
width torch.Size([1, 1, 1, 1])
r shape torch.Size([1, 3, 1, 1])
L 2
tensor([[ 1.0691,  0.1063, -3.4645]], grad_fn=<AddBackward0>)


In [9]:
grad = torch.autograd.grad(gto[0,1], coords, retain_graph=True)
print(grad)

(tensor([[[-0.0000, -0.0000, -0.0000],
         [-0.4897, -0.3220, -0.5255],
         [-0.0000, -0.0000, -0.0000]]]),)


In [10]:
print('coords requires grad', coords.requires_grad)
print('u requires grad', u.requires_grad)
print('s requres grad', s[0].requires_grad)
gto = 0
s_deriv = sph_deriv.spherical_harmonics_deriv(max_L, u)
for L in range(max_L + 1):
    scale = scales[L]
    width = widths[L]
    coeff = coeffs[L]
    sph = s[L].unsqueeze(-1) * coeff
    print('L', L)
    print('coeff shape', coeff.shape)
    print('s[L] shape', s[L].shape)
    print('sph shape', sph.shape)
    print('s[l] deriv shape', s_deriv[L].shape)
    sph_autograd = 0
    if L != 0:
        for i in range(sph.squeeze().shape[0]):
            for j in range(sph.squeeze().shape[1]):
                sph_autograd += torch.autograd.grad(sph.squeeze()[i, j], u, retain_graph=True)[0]
    print('sph autograd', sph_autograd)
    if L != 1:
        sph_grad = (s_deriv[L].unsqueeze(-1) * coeff.unsqueeze(0)).sum((-1, -2))
    else:
        sph_grad = (s_deriv[L].unsqueeze(-1) * coeff[..., [2, 0, 1], :]).sum(-1)
    print('sph deriv', sph_grad) 
    rbf = orbitals.gaussian_rbf(d.unsqueeze(-1), width, scale, L)
    rbf_autograd = 0
    for i in range(3):
        rbf_autograd += torch.autograd.grad(rbf.squeeze()[i], coords, retain_graph=True)[0]
    print('rbf autograd', rbf_autograd)
    rbf_deriv = orbitals.gaussian_rbf_deriv(d.unsqueeze(-1), width, scale, L).squeeze(-1) * coords / d
    print('rbf_deriv', rbf_deriv)
    gto += torch.sum(rbf * sph, dim=(-2, -1))

coords requires grad True
u requires grad True
s requres grad False
4 tensor([[[ 0.7879, -0.2939,  0.5411],
         [ 0.7861, -0.3027, -0.5388],
         [ 0.7370, -0.6460, -0.1991]]], grad_fn=<DivBackward0>)
adding L = 0
adding L = 1
adding L = 2
dxy shape torch.Size([1, 3, 3])
_d3z2m1 shape torch.Size([1, 3, 3])
adding L = 3
dxy shape torch.Size([1, 3, 3])
_d3z2m1 shape torch.Size([1, 3, 3])
adding L = 4
L 0
coeff shape torch.Size([1, 1, 1, 1])
s[L] shape torch.Size([1, 3, 1])
sph shape torch.Size([1, 3, 1, 1])
s[l] deriv shape torch.Size([1, 3, 1])
sph autograd 0
sph deriv tensor([[[0., 0., 0.]]])
rbf autograd tensor([[[ 0.0664, -0.0248,  0.0456],
         [ 0.0491, -0.0189, -0.0336],
         [ 0.0658, -0.0577, -0.0178]]])
rbf_deriv tensor([[[ 0.0664, -0.0248,  0.0456],
         [ 0.0491, -0.0189, -0.0336],
         [ 0.0658, -0.0577, -0.0178]]], grad_fn=<DivBackward0>)
L 1
coeff shape torch.Size([1, 1, 3, 1])
s[L] shape torch.Size([1, 3, 3])
sph shape torch.Size([1, 3, 3, 1])
s[l